# Notebook 1 — Pre-train · Retrain Baselines · CMF Fine-Tune

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026)

This notebook produces the **three checkpoints** that Notebook 2 (unlearning) depends on:

| Stage | Output | Path |
|-------|--------|------|
| **A** | Environment setup & repo clone | — |
| **B** | Configuration | — |
| **C** | Pre-trained model | `checkpoints/pre_train/<dataset>_<arch>_<tag>.pt` |
| **D** | Retrain baselines (one per forget group) | `checkpoints/retrain/<dataset>_<arch>_<tag>/<classes>.pt` |
| **E** | CMF fine-tuned encoder | `checkpoints/CMF_FT_RemoveFC/<dataset>_<arch>_<tag>.pt` |
| **F** | Save `config.json` | `checkpoints/config.json` |

> **After this notebook finishes:**  
> Go to **Output → Add to Dataset** (or the Kaggle dataset page) and create a dataset  
> from `/kaggle/working/checkpoints/`.  
> Then attach that dataset to **Notebook 2** so it can load the checkpoints.

> **Recommended:** GPU T4/P100.  
> Full mode (300-epoch pretrain + 50-epoch retrain × 15 groups) ≈ 5–6 h on T4.  
> Set `TEST_MODE = True` for a ~1 min dataflow check.

## A. Environment Setup

In [ ]:
import subprocess, sys

def sh(cmd, verbose=True):
    """Run shell command, print tail of output, return exit code."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout:
        print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr:
        print("STDERR:", r.stderr[-2000:])
    return r.returncode

sh("pip install -q timm einops scikit-learn matplotlib seaborn")

In [ ]:
import os, sys, json, copy, random, argparse
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')  # keep in sync with latest fixes

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## B. Configuration

| Mode | `TEST_MODE` | What it does | Time |
|------|------------|--------------|------|
| **Test** | `True` | 1% data, 1 epoch, 2 forget groups | ~1 min (CPU) |
| **Full** | `False` | Full data, 300-epoch pretrain, 50-epoch retrain × all groups | ~5–6 h (T4) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  FLIP THIS to False for the real experiment run
TEST_MODE     = True
TEST_FRACTION = 0.01   # 0.001 = 0.1%  |  0.01 = 1%  (only used in TEST_MODE)
# ══════════════════════════════════════════════════════════════════════

# ── Dataset / architecture ────────────────────────────────────────────
DATASET  = 'cifar10'    # 'cifar10' | 'cifar100' | 'tinyimagenet'
ARCH     = {'cifar10': 'resnet18', 'cifar100': 'resnet18',
             'tinyimagenet': 'resnet50'}[DATASET]
IS_VIT   = False
DATA_PATH = '/kaggle/working/data'
SEED      = 1234

# ── Forget-class experiment groups (Appendix A.3) ─────────────────────
if DATASET == 'cifar10':
    SINGLE_CLASS_EXPS = [[0],[1],[2],[3],[4],[5],[6],[7],[8],[9]]
    MULTI_CLASS_EXPS  = [[0,1,2],[3,4,5],[6,7,8],[0,5,9],[2,4,8]]
elif DATASET == 'cifar100':
    SINGLE_CLASS_EXPS = [[0],[1],[2],[3],[5]]
    MULTI_CLASS_EXPS  = [
        [3,15,19,21,31,38,42,43,88,97],
        [47,52,54,56,59,62,70,82,92,96],
        [5,20,22,25,39,40,84,86,87,94],
        [8,13,41,48,59,69,81,85,89,90],
        [1,4,30,32,55,67,72,73,91,95],
    ]
else:  # tinyimagenet
    SINGLE_CLASS_EXPS = [[2],[3],[5],[7],[9]]
    MULTI_CLASS_EXPS  = [
        list(range(0,20)), list(range(20,40)), list(range(40,60)),
        list(range(60,80)), list(range(80,100)),
    ]

ALL_EXPS = ([SINGLE_CLASS_EXPS[0], MULTI_CLASS_EXPS[0]] if TEST_MODE
            else SINGLE_CLASS_EXPS + MULTI_CLASS_EXPS)

# ── Dataset size constants ────────────────────────────────────────────
_TOTAL     = {'cifar10':50000, 'cifar100':50000, 'tinyimagenet':100000}
_PER_CLASS = {'cifar10':5000,  'cifar100':500,   'tinyimagenet':500}

# ── Pre-training hyperparameters ──────────────────────────────────────
if TEST_MODE:
    PRETRAIN_LR, PRETRAIN_EPOCHS, PRETRAIN_BS, PRETRAIN_PATIENCE = 0.05, 1, 8, 1
else:
    PRETRAIN_LR, PRETRAIN_EPOCHS, PRETRAIN_BS, PRETRAIN_PATIENCE = 0.05, 300, 128, 50

# ── Retrain baseline epochs ───────────────────────────────────────────
# 300 × 15 groups ≈ 25 h (too long for Kaggle 12 h limit).
# 50  × 15 groups ≈  4 h (fits comfortably).
# Set RETRAIN_EPOCHS = PRETRAIN_EPOCHS to reproduce the paper exactly.
RETRAIN_EPOCHS = 1 if TEST_MODE else 50

# ── Checkpoint output directory ───────────────────────────────────────
# Everything lands under /kaggle/working/checkpoints/ so Kaggle's
# "Add to Dataset" button can capture the whole tree in one go.
CKPT_ROOT = '/kaggle/working/checkpoints'
os.makedirs(CKPT_ROOT, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

_MODE_TAG = 'test' if TEST_MODE else 'full'

print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'Pretrain epochs={PRETRAIN_EPOCHS}  Retrain epochs={RETRAIN_EPOCHS}')
print(f'Forget groups: {len(ALL_EXPS)}  → {ALL_EXPS}')

## C-helper. Args & Data Helpers

In [ ]:
from utils import get_dataset, get_model, get_retain_forget_partition, test, load_encoder_ckpt_safely
from unlearn import unlear_func
from train import train as train_one_epoch
import utils as _utils_module, functools

# Silence verbose=True default inside unlearn functions
_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw):
    return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=None, class_label_names=None,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=SEED, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=None, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=False, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='paper_repro',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
NUM_CLASSES       = args_base.num_classes
CLASS_LABEL_NAMES = args_base.class_label_names
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    import math, collections

    def _stratified_subset(ds, fraction, seed=SEED):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_class = collections.defaultdict(list)
        for idx, lbl in enumerate(labels):
            by_class[int(lbl)].append(idx)
        kept = []
        for cls_idx in sorted(by_class):
            cls_pool = by_class[cls_idx]
            rng.shuffle(cls_pool)
            n_keep = max(1, math.ceil(len(cls_pool) * fraction))
            kept.extend(cls_pool[:n_keep])
        sub = torch.utils.data.Subset(ds, kept)
        base_targets = ds.targets if hasattr(ds, 'targets') else [
            ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_targets[i] for i in kept]
        return sub

    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)

    _lbls = [int(t) for t in dataset_train.targets]
    _cls_counts = collections.Counter(_lbls)
    _TOTAL[DATASET]     = len(dataset_train)
    _PER_CLASS[DATASET] = max(1, min(_cls_counts.values()))
    print(f'TEST_MODE: {TEST_FRACTION*100:.1f}% → Train={len(dataset_train)}  '
          f'Test={len(dataset_test)}  _TOTAL={_TOTAL[DATASET]}  _PER_CLASS={_PER_CLASS[DATASET]}')
else:
    print(f'Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True,
                 shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, len(dataset_test)), num_workers=2,
                 pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(dataset_train, **LOADER_KW)
test_loader  = torch.utils.data.DataLoader(dataset_test,  **TEST_KW)

## C. Pre-train Original Model (Appendix A.4)

SGD lr=0.05, batch 128, ≤300 epochs, warmup 5, cosine decay, patience 50.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR

CKPT_PRETRAIN = f'{CKPT_ROOT}/pre_train/{DATASET}_{ARCH}_{_MODE_TAG}.pt'
os.makedirs(os.path.dirname(CKPT_PRETRAIN), exist_ok=True)

args_pt = make_args(unlearn_method='pre_train', epochs_or_steps=PRETRAIN_EPOCHS,
                    lr=PRETRAIN_LR, num_classes=NUM_CLASSES,
                    class_label_names=CLASS_LABEL_NAMES, patience=PRETRAIN_PATIENCE)
orig_model = get_model(args_pt, device)

if os.path.exists(CKPT_PRETRAIN):
    print(f'Loading existing checkpoint: {CKPT_PRETRAIN}')
    orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
else:
    print(f'Pre-training {ARCH} on {DATASET} for up to {PRETRAIN_EPOCHS} epochs …')
    total_len = len(dataset_train)
    val_len   = int(total_len * 0.1)
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(total_len, generator=g).tolist()
    tr_loader_pt = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset_train, idx[:total_len-val_len]), **LOADER_KW)
    va_loader_pt = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset_train, idx[total_len-val_len:]), **TEST_KW)

    optimizer = optim.SGD(orig_model.parameters(), lr=PRETRAIN_LR,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)
    _warmup_ms   = min(5, PRETRAIN_EPOCHS)
    warmup_sched = LambdaLR(optimizer, lr_lambda=lambda e: min(1.0, (e+1)/max(1,_warmup_ms)))
    cosine_sched = CosineAnnealingLR(optimizer,
                                     T_max=max(1, PRETRAIN_EPOCHS-_warmup_ms), eta_min=1e-5)
    scheduler    = SequentialLR(optimizer,
                                schedulers=[warmup_sched, cosine_sched],
                                milestones=[_warmup_ms])

    best_acc, no_impr = 0.0, 0
    for epoch in range(1, PRETRAIN_EPOCHS+1):
        train_one_epoch(args_pt, orig_model, device, tr_loader_pt, optimizer, epoch)
        va, _, _ = test(orig_model, device, va_loader_pt, [], CLASS_LABEL_NAMES,
                        NUM_CLASSES, plot_cm=False, job_name='pretrain', set_name='Val')
        scheduler.step()
        if va > best_acc:
            best_acc, no_impr = va, 0
            torch.save(orig_model.state_dict(), CKPT_PRETRAIN)
            print(f'  epoch {epoch:3d}: val={va:.4f} ✓ saved')
        else:
            no_impr += 1
            if no_impr >= PRETRAIN_PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break
    orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))

orig_model.eval()
print('\n── Original model test accuracy ──')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES,
     plot_cm=False, job_name='original', set_name='Test')
print(f'\nCheckpoint saved: {CKPT_PRETRAIN}')

## D. Retrain Baselines — one per forget group (Appendix A.5)

Train from scratch on the retain set only for each forget-class group.  
This is the **gold-standard** upper bound for unlearning methods.

In [ ]:
RETRAIN_RESULTS = []  # {forget_str, retain_acc, forget_acc}

for forget_classes in ALL_EXPS:
    n_forget   = len(forget_classes)
    num_forget = n_forget * _PER_CLASS[DATASET]
    num_retain = _TOTAL[DATASET] - num_forget
    forget_str = ','.join(str(c) for c in forget_classes)
    mode       = 'single' if n_forget == 1 else 'multi'

    print(f'\n{"="*60}')
    print(f'Forget: {forget_classes}  ({mode})')
    print('='*60)

    rt_ckpt = f'{CKPT_ROOT}/retrain/{DATASET}_{ARCH}_{_MODE_TAG}/{forget_str}.pt'
    os.makedirs(os.path.dirname(rt_ckpt), exist_ok=True)

    if not os.path.exists(rt_ckpt):
        retain_ds, _ = get_retain_forget_partition(
            make_args(num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
                      unlearn_class=list(forget_classes)),
            dataset_train, forget_classes)
        rt_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)

        rt_args = make_args(
            unlearn_method='retrain',
            epochs_or_steps=RETRAIN_EPOCHS, lr=PRETRAIN_LR,
            num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
            num_retain_samples=num_retain, num_forget_samples=num_forget,
            unlearn_class=list(forget_classes),
        )
        rt_model = get_model(rt_args, device)
        rt_opt   = optim.SGD(rt_model.parameters(), lr=PRETRAIN_LR,
                             momentum=0.9, weight_decay=5e-4, nesterov=True)
        print(f'  [retrain] {forget_classes} for {RETRAIN_EPOCHS} epochs …')
        rt_model = unlear_func['retrain'](
            args=rt_args, model=rt_model, device=device,
            retain_loader=rt_loader, forget_loader=None,
            train_loader=rt_loader, val_loader=None,
            test_loader=test_loader, optimizer=rt_opt,
            epochs=RETRAIN_EPOCHS, train_dataset=dataset_train,
            val_index=np.arange(len(dataset_train)),
            test_forget_loader=torch.utils.data.DataLoader(dataset_test, **TEST_KW),
        )
        torch.save(rt_model.state_dict(), rt_ckpt)
        print(f'  Saved: {rt_ckpt}')
    else:
        print(f'  Loading cached retrain checkpoint.')
        rt_args = make_args(num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
                            unlearn_class=list(forget_classes))
        rt_model = get_model(rt_args, device)
        rt_model.load_state_dict(torch.load(rt_ckpt, map_location=device))

    rt_model.eval()
    rt_ra, rt_fa, _ = test(rt_model, device, test_loader, forget_classes,
                           CLASS_LABEL_NAMES, NUM_CLASSES,
                           plot_cm=False, job_name='retrain', set_name='Test')
    print(f'  → retain={rt_ra:.4f}  forget={rt_fa:.4f}')
    RETRAIN_RESULTS.append(dict(forget=forget_str, retain_acc=rt_ra, forget_acc=rt_fa))

print('\nAll retrain baselines done.')

## E. CMF Fine-Tune — Prepare CMF Encoder (Appendix B)

Fine-tune the original encoder for 1 epoch with the CMF head on the full dataset.  
This is the starting checkpoint for all CMF-based unlearning methods in Notebook 2.

In [ ]:
CMF_FT_LR     = PRETRAIN_LR if IS_VIT else 1e-3
CMF_FT_EPOCHS = 1  # paper: 1 epoch is sufficient

CKPT_CMF_FT = f'{CKPT_ROOT}/CMF_FT_RemoveFC/{DATASET}_{ARCH}_{_MODE_TAG}.pt'
os.makedirs(os.path.dirname(CKPT_CMF_FT), exist_ok=True)

args_cmf_ft = make_args(
    unlearn_method='CMF_FT_RemoveFC',
    epochs_or_steps=CMF_FT_EPOCHS,
    lr=CMF_FT_LR,
    num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
    num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
    unlearn_class=[], remove_FC=True, CMFClassifier=True,
    weight_decay=1e-4,
)

if os.path.exists(CKPT_CMF_FT):
    print('CMF_FT checkpoint already exists — skipping.')
else:
    cmf_base = get_model(args_cmf_ft, device)
    print('Loading pretrained weights into CMF model …')
    load_encoder_ckpt_safely(cmf_base, CKPT_PRETRAIN, device=str(device))
    opt_ft = optim.SGD(cmf_base.parameters(), lr=CMF_FT_LR,
                       momentum=0.9, weight_decay=1e-4, nesterov=True)
    print('Running CMF_FT_RemoveFC (1 epoch) …')
    cmf_ft_model = unlear_func['CMF_FT_RemoveFC'](
        args=args_cmf_ft, model=cmf_base, device=device,
        retain_loader=train_loader, forget_loader=None,
        train_loader=train_loader, val_loader=None,
        test_loader=test_loader, optimizer=opt_ft,
        epochs=CMF_FT_EPOCHS, train_dataset=dataset_train,
        val_index=np.arange(len(dataset_train)),
        test_forget_loader=torch.utils.data.DataLoader(dataset_test, **TEST_KW),
    )
    torch.save(cmf_ft_model.state_dict(), CKPT_CMF_FT)
    print(f'Saved: {CKPT_CMF_FT}')

print('CMF_FT_RemoveFC base checkpoint ready.')

## F. Save config.json for Notebook 2

`config.json` carries every setting Notebook 2 needs so the two notebooks  
stay in sync without manually copying values.

In [ ]:
import pandas as pd

config = dict(
    # Mode
    TEST_MODE=TEST_MODE,
    TEST_FRACTION=TEST_FRACTION,
    _MODE_TAG=_MODE_TAG,
    # Dataset / arch
    DATASET=DATASET,
    ARCH=ARCH,
    IS_VIT=IS_VIT,
    SEED=SEED,
    # Sizes (after TEST_MODE shrink if applicable)
    NUM_CLASSES=NUM_CLASSES,
    TOTAL=_TOTAL[DATASET],
    PER_CLASS=_PER_CLASS[DATASET],
    # Forget groups
    ALL_EXPS=ALL_EXPS,
    # Training hparams
    PRETRAIN_LR=PRETRAIN_LR,
    PRETRAIN_EPOCHS=PRETRAIN_EPOCHS,
    PRETRAIN_BS=PRETRAIN_BS,
    PRETRAIN_PATIENCE=PRETRAIN_PATIENCE,
    RETRAIN_EPOCHS=RETRAIN_EPOCHS,
    # Checkpoint paths (absolute, inside /kaggle/working/checkpoints/)
    CKPT_PRETRAIN=CKPT_PRETRAIN,
    CKPT_CMF_FT=CKPT_CMF_FT,
    CKPT_ROOT=CKPT_ROOT,
)

config_path = f'{CKPT_ROOT}/config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'config.json saved: {config_path}')

# ── Retrain summary table ──────────────────────────────────────────────
if RETRAIN_RESULTS:
    rt_df = pd.DataFrame(RETRAIN_RESULTS)
    rt_csv = f'{CKPT_ROOT}/retrain_results.csv'
    rt_df.to_csv(rt_csv, index=False)
    print(f'Retrain results saved: {rt_csv}')
    print('\nRetrain summary (mean over forget groups):')
    print(rt_df[['retain_acc','forget_acc']].mean().round(4).to_string())

print('\n' + '='*60)
print('NOTEBOOK 1 COMPLETE')
print('='*60)
print('Checkpoints saved under:', CKPT_ROOT)
print()
print('Next steps:')
print('  1. Kaggle → Output tab → "Add to Dataset"')
print('     (or create a new dataset from /kaggle/working/checkpoints/)')
print('  2. Open Notebook 2 and attach that dataset.')
print('  3. Set CKPT_DATASET_DIR in Notebook 2 to the dataset mount path,')
print('     typically /kaggle/input/<your-dataset-name>/')